In [20]:
# %% [markdown]
# # Additional Analysis Notebook
# ## HIPS, FTIR, and Aethalometer Comparisons
# 
# This notebook implements the additional analyses requested:
# - HIPS Fabs vs Aethalometer attenuation/BCc
# - FTIR-BC vs Aethalometer attenuation/BCc  
# - HIPS-derived MAC vs Aethalometer BCc (colored by PM quartiles)
# - Correlation matrices
# - Time-series ribbon plots
# - Comprehensive correlation heat-maps

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from scipy.stats import pearsonr
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.titlesize': 16
})

# %%
# Update these paths to match your data locations
AETHALOMETER_PATH = "/Users/ahzs645/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/Aethelometry Data/Jacros_MA350_1-min_2022-2024_Cleaned.csv"
DB_PATH = "/Users/ahzs645/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/EC-HIPS-Aeth Comparison/Data/Original Data/Combined Database/spartan_ftir_hips.db"
SPECIATION_CSV_PATH = "/Users/ahzs645/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/EC-HIPS-Aeth Comparison/Data/Downloaded Data/SPARTAN/Addis Ababa/FilterBased_ChemSpecPM25_ETAD.csv"

print("📋 ADDITIONAL ANALYSIS NOTEBOOK")
print("="*60)
print("This notebook will load data from your specified paths and run the additional analyses.")
print(f"Aethalometer data: {AETHALOMETER_PATH}")
print(f"Database: {DB_PATH}")
print(f"Speciation data: {SPECIATION_CSV_PATH}")

# %% [markdown]
# ## Data Loading and Preparation
# 
# Load data using existing functions from your main analysis

# %%
def prepare_comprehensive_dataset(aethalometer_df, overlap_df_all, merged_data):
    """
    Prepare a comprehensive dataset combining all three measurement methods
    """
    print("🔗 Preparing comprehensive dataset...")
    
    # Start with the overlap data (has excellent periods)
    comprehensive_df = overlap_df_all.copy()
    
    # Add aethalometer attenuation data if available
    aethalometer_columns = ['Red ATN1', 'Red ATN2', 'Red BCc', 'Blue BCc', 'Green BCc', 'IR BCc', 'UV BCc']
    available_aeth_cols = [col for col in aethalometer_columns if col in aethalometer_df.columns]
    
    print(f"   Available aethalometer columns: {available_aeth_cols}")
    
    # Map overlap periods to aethalometer data
    for i, row in comprehensive_df.iterrows():
        period_start = row['start_time']
        period_end = row['end_time']
        
        # Extract aethalometer data for this period
        period_aeth = aethalometer_df.loc[period_start:period_end]
        
        if len(period_aeth) > 0:
            # Calculate statistics for each available column
            for col in available_aeth_cols:
                if col in period_aeth.columns:
                    col_data = period_aeth[col].dropna()
                    if len(col_data) > 0:
                        comprehensive_df.loc[i, f'aeth_{col.replace(" ", "_").lower()}_mean'] = col_data.mean()
                        comprehensive_df.loc[i, f'aeth_{col.replace(" ", "_").lower()}_std'] = col_data.std()
                        comprehensive_df.loc[i, f'aeth_{col.replace(" ", "_").lower()}_median'] = col_data.median()
    
    # Add chemical speciation data from merged_data if available
    speciation_columns = ['PM25_mass', 'Iron', 'Potassium_Ion', 'Sulfate_Ion', 'Nitrate_Ion', 
                         'Ammonium_Ion', 'Aluminum', 'Manganese', 'Zinc', 'Copper']
    
    # Try to match dates between comprehensive_df and merged_data
    if merged_data is not None and len(merged_data) > 0:
        print("   Adding chemical speciation data...")
        
        for spec_col in speciation_columns:
            if spec_col in merged_data.columns:
                comprehensive_df[spec_col] = np.nan
                
                # Match by filter_id if available
                if 'filter_id' in comprehensive_df.columns and 'filter_id' in merged_data.columns:
                    for i, row in comprehensive_df.iterrows():
                        if pd.notna(row['filter_id']):
                            matching_spec = merged_data[merged_data['filter_id'] == row['filter_id']]
                            if len(matching_spec) > 0:
                                comprehensive_df.loc[i, spec_col] = matching_spec[spec_col].iloc[0]
                
                # If no filter_id match, try date matching
                else:
                    for i, row in comprehensive_df.iterrows():
                        sample_date = row['start_time'].date()
                        
                        # Try different date columns in merged_data
                        date_cols = ['sample_date', 'Start_Date', 'date']
                        for date_col in date_cols:
                            if date_col in merged_data.columns:
                                matching_spec = merged_data[merged_data[date_col].dt.date == sample_date]
                                if len(matching_spec) > 0:
                                    comprehensive_df.loc[i, spec_col] = matching_spec[spec_col].iloc[0]
                                    break
    
    print(f"✅ Comprehensive dataset prepared: {len(comprehensive_df)} samples")
    print(f"   Columns: {len(comprehensive_df.columns)}")
    
    return comprehensive_df

# %%
# Example of how to load data (adjust based on your actual data loading)
# This assumes you've already run your main analysis and have the data loaded

# If you need to load fresh data, uncomment and modify these lines:
# from your_main_analysis_script import load_aethalometer_data, load_filter_sample_data, merge_etad_datasets
# aethalometer_df = load_aethalometer_data(AETHALOMETER_PATH, convert_to_ug=True)
# etad_data, ftir_data = load_filter_sample_data(DB_PATH)
# merged_data = merge_etad_datasets(ftir_data, speciation_data)

# For now, we'll assume data is already loaded
print("📊 Assuming data is already loaded from main analysis...")
print("   Variables expected: aethalometer_df, overlap_df_all, merged_data")

# %% [markdown]
# ## Analysis 4: HIPS Fabs vs Aethalometer Attenuation/BCc

# %%
def plot_hips_vs_aethalometer(comprehensive_df, use_attenuation=True, wavelength='red'):
    """
    Plot HIPS Fabs vs Aethalometer attenuation or BCc
    
    Parameters:
    -----------
    comprehensive_df : pandas.DataFrame
        Combined dataset
    use_attenuation : bool
        If True, use ATN1; if False, use BCc
    wavelength : str
        Wavelength to use ('red', 'blue', 'green', 'ir', 'uv')
    """
    
    print(f"📈 Analysis 4: HIPS Fabs vs Aethalometer {'Attenuation' if use_attenuation else 'BCc'}")
    
    # Determine which aethalometer column to use
    if use_attenuation:
        aeth_col = f'aeth_{wavelength}_atn1_mean'
        aeth_label = f'{wavelength.title()} ATN1'
        aeth_units = 'ATN units'
    else:
        aeth_col = f'aeth_{wavelength}_bcc_mean'
        aeth_label = f'{wavelength.title()} BCc'
        aeth_units = 'μg/m³'
    
    # Check if columns exist
    if aeth_col not in comprehensive_df.columns:
        print(f"❌ Column {aeth_col} not found. Available columns:")
        available = [col for col in comprehensive_df.columns if 'aeth_' in col]
        for col in available[:10]:  # Show first 10
            print(f"   - {col}")
        return None
    
    if 'Fabs' not in comprehensive_df.columns:
        print(f"❌ Fabs column not found")
        return None
    
    # Prepare data
    df_clean = comprehensive_df[[aeth_col, 'Fabs']].dropna()
    
    if len(df_clean) < 5:
        print(f"❌ Insufficient data: only {len(df_clean)} samples")
        return None
    
    x_data = df_clean[aeth_col]
    y_data = df_clean['Fabs']
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Check for seasonal data
    if 'season' in comprehensive_df.columns:
        seasons_data = comprehensive_df.loc[df_clean.index, 'season'].dropna()
        
        if len(seasons_data) > 0:
            season_colors = {
                'Dry Season (Bega)': '#d35400', 
                'Belg Rainy Season': '#27ae60', 
                'Kiremt Rainy Season': '#2980b9'
            }
            
            print(f"   🌍 Adding seasonal color coding")
            
            for season in seasons_data.unique():
                if pd.isna(season):
                    continue
                    
                season_mask = comprehensive_df.loc[df_clean.index, 'season'] == season
                if season_mask.sum() < 3:
                    continue
                
                season_x = x_data[season_mask]
                season_y = y_data[season_mask]
                
                if season in season_colors and len(season_x) >= 3:
                    ax.scatter(season_x, season_y, 
                              color=season_colors[season], alpha=0.7, s=60, 
                              label=season.split()[0], edgecolors='black', linewidth=1)
                    
                    # Individual regression line
                    if len(season_x) >= 5:
                        z_season = np.polyfit(season_x, season_y, 1)
                        p_season = np.poly1d(z_season)
                        x_line_season = np.linspace(season_x.min(), season_x.max(), 50)
                        ax.plot(x_line_season, p_season(x_line_season), 
                               color=season_colors[season], linewidth=2, 
                               linestyle='--', alpha=0.9)
        else:
            # No seasonal data
            ax.scatter(x_data, y_data, alpha=0.7, s=60, color='blue', 
                      edgecolors='black', linewidth=1)
    else:
        # No seasonal column
        ax.scatter(x_data, y_data, alpha=0.7, s=60, color='blue', 
                  edgecolors='black', linewidth=1)
    
    # Overall regression line
    if len(x_data) > 5:
        z_overall = np.polyfit(x_data, y_data, 1)
        p_overall = np.poly1d(z_overall)
        x_line_overall = np.linspace(x_data.min(), x_data.max(), 100)
        ax.plot(x_line_overall, p_overall(x_line_overall), 'black', linewidth=3, 
               linestyle='-', alpha=0.8, label='Overall Trend')
        
        # Statistics
        r_overall, p_overall = pearsonr(x_data, y_data)
        
        # Add interpretation based on literature
        if r_overall >= 0.8:
            interp = "Excellent"
        elif r_overall >= 0.6:
            interp = "Good"
        elif r_overall >= 0.4:
            interp = "Moderate"
        else:
            interp = "Poor"
        
        ax.text(0.05, 0.95, f'r = {r_overall:.3f} ({interp})\np = {p_overall:.2e}\nn = {len(x_data)}', 
               transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               fontweight='bold', va='top')
    
    ax.set_xlabel(f'Aethalometer {aeth_label} ({aeth_units})', fontweight='bold')
    ax.set_ylabel('HIPS Fabs (Mm⁻¹)', fontweight='bold')
    ax.set_title(f'HIPS Fabs vs Aethalometer {aeth_label}\n(n={len(x_data)} samples)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if 'season' in comprehensive_df.columns:
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    return r_overall if 'r_overall' in locals() else None

# %% [markdown]
# ## Analysis 5: FTIR-BC vs Aethalometer Attenuation/BCc

# %%
def plot_ftir_vs_aethalometer(comprehensive_df, use_attenuation=True, wavelength='red'):
    """
    Plot FTIR-BC vs Aethalometer attenuation or BCc
    """
    
    print(f"📈 Analysis 5: FTIR-BC vs Aethalometer {'Attenuation' if use_attenuation else 'BCc'}")
    
    # Determine which aethalometer column to use
    if use_attenuation:
        aeth_col = f'aeth_{wavelength}_atn1_mean'
        aeth_label = f'{wavelength.title()} ATN1'
        aeth_units = 'ATN units'
    else:
        aeth_col = f'aeth_{wavelength}_bcc_mean'
        aeth_label = f'{wavelength.title()} BCc'
        aeth_units = 'μg/m³'
    
    # Check for FTIR-BC column
    ftir_col = 'EC_FTIR'
    if ftir_col not in comprehensive_df.columns:
        print(f"❌ {ftir_col} column not found")
        return None
    
    if aeth_col not in comprehensive_df.columns:
        print(f"❌ {aeth_col} column not found")
        return None
    
    # Prepare data
    df_clean = comprehensive_df[[aeth_col, ftir_col]].dropna()
    
    if len(df_clean) < 5:
        print(f"❌ Insufficient data: only {len(df_clean)} samples")
        return None
    
    x_data = df_clean[aeth_col]
    y_data = df_clean[ftir_col]
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Check for seasonal data
    if 'season' in comprehensive_df.columns:
        seasons_data = comprehensive_df.loc[df_clean.index, 'season'].dropna()
        
        if len(seasons_data) > 0:
            season_colors = {
                'Dry Season (Bega)': '#d35400', 
                'Belg Rainy Season': '#27ae60', 
                'Kiremt Rainy Season': '#2980b9'
            }
            
            print(f"   🌍 Adding seasonal color coding")
            
            for season in seasons_data.unique():
                if pd.isna(season):
                    continue
                    
                season_mask = comprehensive_df.loc[df_clean.index, 'season'] == season
                if season_mask.sum() < 3:
                    continue
                
                season_x = x_data[season_mask]
                season_y = y_data[season_mask]
                
                if season in season_colors and len(season_x) >= 3:
                    ax.scatter(season_x, season_y, 
                              color=season_colors[season], alpha=0.7, s=60, 
                              label=season.split()[0], edgecolors='black', linewidth=1)
                    
                    # Individual regression line
                    if len(season_x) >= 5:
                        z_season = np.polyfit(season_x, season_y, 1)
                        p_season = np.poly1d(z_season)
                        x_line_season = np.linspace(season_x.min(), season_x.max(), 50)
                        ax.plot(x_line_season, p_season(x_line_season), 
                               color=season_colors[season], linewidth=2, 
                               linestyle='--', alpha=0.9)
        else:
            ax.scatter(x_data, y_data, alpha=0.7, s=60, color='red', 
                      edgecolors='black', linewidth=1)
    else:
        ax.scatter(x_data, y_data, alpha=0.7, s=60, color='red', 
                  edgecolors='black', linewidth=1)
    
    # Overall regression line
    if len(x_data) > 5:
        z_overall = np.polyfit(x_data, y_data, 1)
        p_overall = np.poly1d(z_overall)
        x_line_overall = np.linspace(x_data.min(), x_data.max(), 100)
        ax.plot(x_line_overall, p_overall(x_line_overall), 'black', linewidth=3, 
               linestyle='-', alpha=0.8, label='Overall Trend')
        
        # Statistics
        r_overall, p_overall = pearsonr(x_data, y_data)
        
        # Add interpretation based on literature
        if r_overall >= 0.8:
            interp = "Excellent"
        elif r_overall >= 0.6:
            interp = "Good"
        elif r_overall >= 0.4:
            interp = "Moderate"
        else:
            interp = "Poor"
        
        ax.text(0.05, 0.95, f'r = {r_overall:.3f} ({interp})\np = {p_overall:.2e}\nn = {len(x_data)}', 
               transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               fontweight='bold', va='top')
    
    ax.set_xlabel(f'Aethalometer {aeth_label} ({aeth_units})', fontweight='bold')
    ax.set_ylabel('FTIR-BC (μg/m³)', fontweight='bold')
    ax.set_title(f'FTIR-BC vs Aethalometer {aeth_label}\n(n={len(x_data)} samples)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if 'season' in comprehensive_df.columns:
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    return r_overall if 'r_overall' in locals() else None

# %% [markdown]
# ## Analysis 6: HIPS-derived MAC vs Aethalometer BCc (colored by PM quartiles)

# %%
def plot_mac_vs_aethalometer_pm_quartiles(comprehensive_df, wavelength='red'):
    """
    Plot HIPS-derived MAC vs Aethalometer BCc, colored by PM loading quartiles
    """
    
    print(f"📈 Analysis 6: HIPS-derived MAC vs Aethalometer BCc (colored by PM quartiles)")
    
    aeth_col = f'aeth_{wavelength}_bcc_mean'
    
    # Check required columns
    required_cols = [aeth_col, 'Fabs', 'EC_FTIR']
    missing_cols = [col for col in required_cols if col not in comprehensive_df.columns]
    
    if missing_cols:
        print(f"❌ Missing required columns: {missing_cols}")
        return None
    
    # Calculate HIPS-derived MAC
    comprehensive_df = comprehensive_df.copy()
    comprehensive_df['HIPS_MAC'] = comprehensive_df['Fabs'] / comprehensive_df['EC_FTIR']
    
    # Prepare data (filter reasonable MAC values)
    df_clean = comprehensive_df[[aeth_col, 'HIPS_MAC', 'PM25_mass']].dropna()
    reasonable_mask = (df_clean['HIPS_MAC'] > 5) & (df_clean['HIPS_MAC'] < 25)
    df_clean = df_clean[reasonable_mask]
    
    if len(df_clean) < 10:
        print(f"❌ Insufficient data: only {len(df_clean)} samples")
        return None
    
    # Create PM2.5 quartiles
    pm_quartiles = pd.qcut(df_clean['PM25_mass'], q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
    
    x_data = df_clean[aeth_col]
    y_data = df_clean['HIPS_MAC']
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Color by PM quartiles
    quartile_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
    
    print(f"   📊 PM2.5 quartiles:")
    for i, quartile in enumerate(pm_quartiles.cat.categories):
        quartile_mask = pm_quartiles == quartile
        quartile_data = df_clean[quartile_mask]
        
        if len(quartile_data) >= 3:
            quartile_x = quartile_data[aeth_col]
            quartile_y = quartile_data['HIPS_MAC']
            pm_range = f"{quartile_data['PM25_mass'].min():.1f}-{quartile_data['PM25_mass'].max():.1f}"
            
            ax.scatter(quartile_x, quartile_y, 
                      color=quartile_colors[i], alpha=0.7, s=60, 
                      label=f'{quartile}\n({pm_range} μg/m³)', 
                      edgecolors='black', linewidth=1)
            
            print(f"     {quartile}: {pm_range} μg/m³, n={len(quartile_data)}")
            
            # Individual regression line for each quartile
            if len(quartile_x) >= 5:
                z_quartile = np.polyfit(quartile_x, quartile_y, 1)
                p_quartile = np.poly1d(z_quartile)
                x_line_quartile = np.linspace(quartile_x.min(), quartile_x.max(), 50)
                ax.plot(x_line_quartile, p_quartile(x_line_quartile), 
                       color=quartile_colors[i], linewidth=2, 
                       linestyle='--', alpha=0.8)
    
    # Overall regression line
    if len(x_data) > 5:
        z_overall = np.polyfit(x_data, y_data, 1)
        p_overall = np.poly1d(z_overall)
        x_line_overall = np.linspace(x_data.min(), x_data.max(), 100)
        ax.plot(x_line_overall, p_overall(x_line_overall), 'black', linewidth=3, 
               linestyle='-', alpha=0.8, label='Overall Trend')
        
        # Statistics
        r_overall, p_overall = pearsonr(x_data, y_data)
        
        # Add interpretation based on literature
        if r_overall >= 0.8:
            interp = "Excellent"
        elif r_overall >= 0.6:
            interp = "Good"
        elif r_overall >= 0.4:
            interp = "Moderate"
        else:
            interp = "Poor"
        
        ax.text(0.05, 0.95, f'Overall: r = {r_overall:.3f} ({interp})\np = {p_overall:.2e}\nn = {len(x_data)}', 
               transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               fontweight='bold', va='top')
    
    ax.set_xlabel(f'Aethalometer {wavelength.title()} BCc (μg/m³)', fontweight='bold')
    ax.set_ylabel('HIPS-derived MAC (m²/g)', fontweight='bold')
    ax.set_title(f'HIPS MAC vs Aethalometer BCc\n(Colored by PM₂.₅ Loading Quartiles)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
    return r_overall if 'r_overall' in locals() else None

# %% [markdown]
# ## Analysis 7: Correlation Matrix

# %%
def create_correlation_matrix(comprehensive_df):
    """
    Create correlation matrix of key variables
    """
    
    print(f"📊 Analysis 7: Correlation Matrix")
    
    # Define key variables for correlation matrix
    key_variables = []
    variable_labels = {}
    
    # Aethalometer variables
    aeth_vars = ['aeth_red_atn1_mean', 'aeth_red_bcc_mean', 'aeth_blue_bcc_mean', 'aeth_green_bcc_mean']
    for var in aeth_vars:
        if var in comprehensive_df.columns:
            key_variables.append(var)
            if 'atn1' in var:
                variable_labels[var] = 'Aeth ATN1'
            elif 'red_bcc' in var:
                variable_labels[var] = 'Aeth Red BCc'
            elif 'blue_bcc' in var:
                variable_labels[var] = 'Aeth Blue BCc'
            elif 'green_bcc' in var:
                variable_labels[var] = 'Aeth Green BCc'
    
    # HIPS and FTIR variables
    if 'Fabs' in comprehensive_df.columns:
        key_variables.append('Fabs')
        variable_labels['Fabs'] = 'HIPS Fabs'
    
    if 'EC_FTIR' in comprehensive_df.columns:
        key_variables.append('EC_FTIR')
        variable_labels['EC_FTIR'] = 'FTIR-BC'
    
    # Chemical species
    chem_vars = ['Iron', 'Potassium_Ion', 'PM25_mass', 'Sulfate_Ion', 'Aluminum']
    for var in chem_vars:
        if var in comprehensive_df.columns:
            key_variables.append(var)
            if var == 'Potassium_Ion':
                variable_labels[var] = 'K⁺'
            elif var == 'PM25_mass':
                variable_labels[var] = 'PM₂.₅ Mass'
            elif var == 'Sulfate_Ion':
                variable_labels[var] = 'SO₄²⁻'
            else:
                variable_labels[var] = var
    
    if len(key_variables) < 3:
        print(f"❌ Insufficient variables for correlation matrix: {key_variables}")
        return None
    
    print(f"   Variables for correlation matrix: {len(key_variables)}")
    for var in key_variables:
        print(f"     - {variable_labels.get(var, var)}")
    
    # Calculate correlation matrix
    corr_data = comprehensive_df[key_variables].dropna()
    
    if len(corr_data) < 5:
        print(f"❌ Insufficient complete cases: {len(corr_data)}")
        return None
    
    corr_matrix = corr_data.corr()
    
    # Create heatmap
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Use variable labels for display
    display_labels = [variable_labels.get(var, var) for var in key_variables]
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Hide upper triangle
    
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                square=True, fmt='.3f', cbar_kws={"shrink": .8},
                xticklabels=display_labels, yticklabels=display_labels, ax=ax)
    
    ax.set_title(f'Correlation Matrix (R values)\n(n={len(corr_data)} complete cases)', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return corr_matrix

# %% [markdown]
# ## Additional Analysis 2: Time-series Ribbon Plot

# %%
def create_timeseries_ribbon_plot(comprehensive_df):
    """
    Create time-series ribbon plot with daily values of three BC estimates
    """
    
    print(f"📈 Additional Analysis 2: Time-series Ribbon Plot")
    
    # Check required columns
    bc_columns = {}
    if 'aeth_red_bcc_mean' in comprehensive_df.columns:
        bc_columns['Aethalometer'] = 'aeth_red_bcc_mean'
    if 'EC_FTIR' in comprehensive_df.columns:
        bc_columns['FTIR'] = 'EC_FTIR'
    if 'Fabs' in comprehensive_df.columns and 'EC_FTIR' in comprehensive_df.columns:
        # Calculate HIPS-derived BC using a typical MAC value
        comprehensive_df['HIPS_BC'] = comprehensive_df['Fabs'] / 10.0  # Assuming MAC = 10 m²/g
        bc_columns['HIPS'] = 'HIPS_BC'
    
    if len(bc_columns) < 2:
        print(f"❌ Need at least 2 BC estimates. Available: {list(bc_columns.keys())}")
        return None
    
    # Prepare time series data
    df_ts = comprehensive_df.copy()
    df_ts['date'] = df_ts['start_time'].dt.date
    
    # Calculate daily statistics for each method
    daily_stats = {}
    
    for method, column in bc_columns.items():
        method_data = df_ts[[column, 'date']].dropna()
        daily_method = method_data.groupby('date')[column].agg(['mean', 'std']).reset_index()
        daily_method['method'] = method
        daily_stats[method] = daily_method
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    
    colors = {'Aethalometer': '#e74c3c', 'FTIR': '#2980b9', 'HIPS': '#27ae60'}
    
    for method, daily_data in daily_stats.items():
        dates = pd.to_datetime(daily_data['date'])
        means = daily_data['mean']
        stds = daily_data['std'].fillna(0)  # Fill NaN std with 0
        
        color = colors.get(method, 'gray')
        
        # Plot mean line
        ax.plot(dates, means, color=color, linewidth=2, label=f'{method} BC', alpha=0.8)
        
        # Plot ±1σ ribbon
        ax.fill_between(dates, means - stds, means + stds, 
                       color=color, alpha=0.2, label=f'{method} ±1σ')
    
    ax.set_xlabel('Date', fontweight='bold')
    ax.set_ylabel('BC Concentration (μg/m³)', fontweight='bold')
    ax.set_title('Time Series Comparison of BC Estimates\n(Daily values with ±1σ bands)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Rotate x-axis labels
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    return daily_stats

# %% [markdown]
# ## Additional Analysis 7: Comprehensive Correlation Heat-map

# %%
def create_comprehensive_correlation_heatmap(comprehensive_df):
    """
    Create comprehensive correlation heat-map of all major elements + ions + BC estimates
    """
    
    print(f"🔥 Additional Analysis 7: Comprehensive Correlation Heat-map")
    
    # Define comprehensive variable list
    all_variables = []
    variable_categories = {}
    
    # BC estimates
    bc_vars = ['aeth_red_bcc_mean', 'aeth_blue_bcc_mean', 'EC_FTIR', 'Fabs']
    for var in bc_vars:
        if var in comprehensive_df.columns:
            all_variables.append(var)
            variable_categories[var] = 'BC_Estimates'
    
    # Major elements
    elements = ['Iron', 'Aluminum', 'Silicon', 'Titanium', 'Manganese', 'Zinc', 'Copper', 'Lead']
    for var in elements:
        if var in comprehensive_df.columns:
            all_variables.append(var)
            variable_categories[var] = 'Elements'
    
    # Ions
    ions = ['Potassium_Ion', 'Sulfate_Ion', 'Nitrate_Ion', 'Ammonium_Ion', 'Sodium_Ion', 'Calcium_Ion']
    for var in ions:
        if var in comprehensive_df.columns:
            all_variables.append(var)
            variable_categories[var] = 'Ions'
    
    # PM mass
    if 'PM25_mass' in comprehensive_df.columns:
        all_variables.append('PM25_mass')
        variable_categories['PM25_mass'] = 'PM_Mass'
    
    if len(all_variables) < 5:
        print(f"❌ Insufficient variables for comprehensive heatmap: {all_variables}")
        return None
    
    print(f"   Variables for comprehensive correlation: {len(all_variables)}")
    
    # Prepare data with log transformation where needed
    corr_data = comprehensive_df[all_variables].copy()
    
    # Apply log transformation to highly skewed variables
    log_transform_vars = []
    for var in all_variables:
        if var in comprehensive_df.columns:
            data_series = comprehensive_df[var].dropna()
            if len(data_series) > 5:
                # Check if data is positive and has high skewness
                if (data_series > 0).all() and data_series.std() / data_series.mean() > 1:
                    corr_data[var] = np.log10(data_series + 1e-6)  # Add small constant to avoid log(0)
                    log_transform_vars.append(var)
    
    # Remove rows with any NaN values
    corr_data = corr_data.dropna()
    
    if len(corr_data) < 5:
        print(f"❌ Insufficient complete cases: {len(corr_data)}")
        return None
    
    print(f"   Complete cases: {len(corr_data)}")
    print(f"   Log-transformed variables: {log_transform_vars}")
    
    # Calculate correlation matrix
    corr_matrix = corr_data.corr()
    
    # Create enhanced heatmap
    fig, ax = plt.subplots(1, 1, figsize=(14, 12))
    
    # Create custom colormap
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Hide upper triangle
    
    # Create heatmap
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                square=True, fmt='.2f', cbar_kws={"shrink": .8},
                xticklabels=True, yticklabels=True, ax=ax)
    
    # Customize labels
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    
    ax.set_title(f'Comprehensive Correlation Matrix\nAll Major Elements + Ions + BC Estimates\n(n={len(corr_data)}, log-scaled where needed)', 
                 fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Print strong correlations
    print(f"\n🔍 STRONG CORRELATIONS (|r| > 0.7):")
    strong_correlations = []
    
    for i, var1 in enumerate(corr_matrix.columns):
        for j, var2 in enumerate(corr_matrix.columns):
            if i < j:  # Only consider lower triangle
                r_val = corr_matrix.loc[var1, var2]
                if abs(r_val) > 0.7:
                    strong_correlations.append((var1, var2, r_val))
    
    strong_correlations.sort(key=lambda x: abs(x[2]), reverse=True)
    
    for var1, var2, r_val in strong_correlations[:10]:  # Top 10
        print(f"   {var1} ↔ {var2}: r = {r_val:.3f}")
    
    return corr_matrix, variable_categories

# %% [markdown]
# ## Main Execution Function

# %%
def run_additional_analyses(aethalometer_df=None, overlap_df_all=None, merged_data=None):
    """
    Run all additional analyses
    """
    
    print("🚀 RUNNING ADDITIONAL ANALYSES")
    print("="*60)
    
    # Check if data is provided or needs to be loaded
    if aethalometer_df is None or overlap_df_all is None:
        print("❌ Required data not provided. Please load:")
        print("   - aethalometer_df")
        print("   - overlap_df_all") 
        print("   - merged_data (optional but recommended)")
        return None
    
    # Prepare comprehensive dataset
    comprehensive_df = prepare_comprehensive_dataset(aethalometer_df, overlap_df_all, merged_data)
    
    print(f"\n📊 Dataset prepared with {len(comprehensive_df)} samples")
    print(f"   Available columns: {comprehensive_df.columns.tolist()}")
    
    # Run all analyses
    results = {}
    
    print(f"\n" + "="*60)
    print("ANALYSIS 4: HIPS Fabs vs Aethalometer")
    results['hips_vs_aeth_atn'] = plot_hips_vs_aethalometer(comprehensive_df, use_attenuation=True)
    results['hips_vs_aeth_bcc'] = plot_hips_vs_aethalometer(comprehensive_df, use_attenuation=False)
    
    print(f"\n" + "="*60)
    print("ANALYSIS 5: FTIR-BC vs Aethalometer")
    results['ftir_vs_aeth_atn'] = plot_ftir_vs_aethalometer(comprehensive_df, use_attenuation=True)
    results['ftir_vs_aeth_bcc'] = plot_ftir_vs_aethalometer(comprehensive_df, use_attenuation=False)
    
    print(f"\n" + "="*60)
    print("ANALYSIS 6: HIPS MAC vs Aethalometer (PM quartiles)")
    results['mac_vs_aeth_pm'] = plot_mac_vs_aethalometer_pm_quartiles(comprehensive_df)
    
    print(f"\n" + "="*60)
    print("ANALYSIS 7: Correlation Matrix")
    results['correlation_matrix'] = create_correlation_matrix(comprehensive_df)
    
    print(f"\n" + "="*60)
    print("ADDITIONAL ANALYSIS 2: Time-series Ribbon Plot")
    results['timeseries_ribbons'] = create_timeseries_ribbon_plot(comprehensive_df)
    
    print(f"\n" + "="*60)
    print("ADDITIONAL ANALYSIS 7: Comprehensive Correlation Heat-map")
    results['comprehensive_heatmap'] = create_comprehensive_correlation_heatmap(comprehensive_df)
    
    print(f"\n✅ ALL ADDITIONAL ANALYSES COMPLETE!")
    
    return results, comprehensive_df

# %% [markdown]
# ## Example Usage
# 
# Run the analyses using your existing data:

# %%
# Example usage (uncomment and modify as needed):

# # Method 1: If you have the data already loaded
# # additional_results, comprehensive_dataset = run_additional_analyses(
# #     aethalometer_df=your_aethalometer_df,
# #     overlap_df_all=your_overlap_df_all,
# #     merged_data=your_merged_data
# # )

# # Method 2: Load fresh data (modify paths)
# if False:  # Set to True to run data loading
#     # Load aethalometer data
#     from your_main_script import load_aethalometer_data
#     aethalometer_df = load_aethalometer_data(AETHALOMETER_PATH, convert_to_ug=True)
    
#     # Load and process overlap data
#     from your_main_script import load_filter_sample_data, find_overlapping_excellent_data
#     etad_data, ftir_data = load_filter_sample_data(DB_PATH)
#     # ... additional processing steps
    
#     # Run analyses
#     additional_results, comprehensive_dataset = run_additional_analyses(
#         aethalometer_df, overlap_df_all, merged_data
#     )

print("📝 TO RUN THE ANALYSES:")
print("1. Make sure your data is loaded (aethalometer_df, overlap_df_all, merged_data)")
print("2. Run: additional_results, comprehensive_dataset = run_additional_analyses(aethalometer_df, overlap_df_all, merged_data)")
print("\nThis will create all 6 requested analyses plus the time-series ribbon plot and comprehensive correlation heatmap!")

# %% [markdown]
# ## Important Considerations for BC Method Comparisons
# 
# Based on recent research findings correlations between aethalometer, FTIR, and HIPS measurements can vary significantly depending on several factors:
# 
# **Instrument-Specific Factors:**
# - Dual-spot aethalometers (AE33) provide real-time loading compensation, improving accuracy compared to older models
# - Aethalometer values can be only 23% of integrating sphere values for pure BC aerosols, indicating calibration dependencies
# - Different aethalometer models can show ~9% differences in BC concentrations
# 
# **Environmental Dependencies:**
# - Correlation slopes between thermal EC and optical BC varied from 3.3 to 2.7 in different cities, and dropped to 0.4 during forest fire events
# - Absorption Ångström exponents significantly affect source apportionment results and correlations
# - Seasonal variations in aerosol composition affect spectral dependence of absorption
# 
# **Interpretation Guidelines:**
# 1. **Expected correlation ranges**: r = 0.6-0.9 for well-calibrated instruments under stable conditions
# 2. **Seasonal effects**: Expect lower correlations during biomass burning episodes or dust events
# 3. **Loading effects**: Monitor for systematic biases, especially at high concentrations
# 4. **Wavelength dependence**: Red channel (635/660nm) typically shows best correlation with thermal methods
# 
# # %% [markdown]
# ## Summary
# 
# This notebook provides:
# 
# **Core Analyses (4-7):**
# - **Analysis 4**: HIPS Fabs vs Aethalometer (both attenuation and BCc)
# - **Analysis 5**: FTIR-BC vs Aethalometer (both attenuation and BCc)  
# - **Analysis 6**: HIPS-derived MAC vs Aethalometer BCc (colored by PM quartiles)
# - **Analysis 7**: Key variable correlation matrix
# 
# **Additional Analyses:**
# - **Time-series ribbon plot**: Daily BC estimates with ±1σ bands
# - **Comprehensive correlation heatmap**: All elements + ions + BC estimates
# 
# **Features:**
# - Seasonal color coding where available
# - Statistical reporting (correlations, p-values)
# - Robust data handling and error checking
# - Publication-quality visualizations
# - Comprehensive logging and status updates
# - Literature-based interpretation guidelines
# 
# **Key Questions Addressed:**
# - How do different BC measurement methods correlate?
# - What role does PM mass loading play in MAC variability?
# - Are there seasonal patterns in method agreement?
# - Which chemical species are most strongly correlated?
# - When do methods agree/disagree over time?
# - What are reasonable correlation expectations based on current literature?

📋 ADDITIONAL ANALYSIS NOTEBOOK
This notebook will load data from your specified paths and run the additional analyses.
Aethalometer data: /Users/ahzs645/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/Aethelometry Data/Jacros_MA350_1-min_2022-2024_Cleaned.csv
Database: /Users/ahzs645/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/EC-HIPS-Aeth Comparison/Data/Original Data/Combined Database/spartan_ftir_hips.db
Speciation data: /Users/ahzs645/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/EC-HIPS-Aeth Comparison/Data/Downloaded Data/SPARTAN/Addis Ababa/FilterBased_ChemSpecPM25_ETAD.csv
📊 Assuming data is already loaded from main analysis...
   Variables expected: aethalometer_df, overlap_df_all, merged_data
📝 TO RUN THE ANALYSES:
1. Make sure your data is loaded (aethalometer_df, overlap_df_al

In [18]:
from your_main_script import load_aethalometer_data, load_filter_sample_data, merge_etad_datasets

aethalometer_df = load_aethalometer_data(AETHALOMETER_PATH, convert_to_ug=True)
etad_data, ftir_data = load_filter_sample_data(DB_PATH)
# Assuming you have a merged speciation CSV loaded into `speciation_data`
merged_data = merge_etad_datasets(ftir_data, speciation_data)

# Generate overlap_df_all (using your overlap method)
# overlap_df_all = find_overlapping_excellent_data(...)  # replace with your actual method

ModuleNotFoundError: No module named 'your_main_script'

In [14]:
# %% [markdown]
# ## Main Execution Function

# %%
def run_additional_analyses(aethalometer_df=None, overlap_df_all=None, merged_data=None):
    """
    Run all additional analyses
    """
    
    print("🚀 RUNNING ADDITIONAL ANALYSES")
    print("="*60)
    
    # Check if data is provided or needs to be loaded
    if aethalometer_df is None or overlap_df_all is None:
        print("❌ Required data not provided. Please load:")
        print("   - aethalometer_df")
        print("   - overlap_df_all") 
        print("   - merged_data (optional but recommended)")
        return None
    
    # Prepare comprehensive dataset
    comprehensive_df = prepare_comprehensive_dataset(aethalometer_df, overlap_df_all, merged_data)
    
    print(f"\n📊 Dataset prepared with {len(comprehensive_df)} samples")
    print(f"   Available columns: {comprehensive_df.columns.tolist()}")
    
    # Run all analyses
    results = {}
    
    print(f"\n" + "="*60)
    print("ANALYSIS 4: HIPS Fabs vs Aethalometer")
    results['hips_vs_aeth_atn'] = plot_hips_vs_aethalometer(comprehensive_df, use_attenuation=True)
    results['hips_vs_aeth_bcc'] = plot_hips_vs_aethalometer(comprehensive_df, use_attenuation=False)
    
    print(f"\n" + "="*60)
    print("ANALYSIS 5: FTIR-BC vs Aethalometer")
    results['ftir_vs_aeth_atn'] = plot_ftir_vs_aethalometer(comprehensive_df, use_attenuation=True)
    results['ftir_vs_aeth_bcc'] = plot_ftir_vs_aethalometer(comprehensive_df, use_attenuation=False)
    
    print(f"\n" + "="*60)
    print("ANALYSIS 6: HIPS MAC vs Aethalometer (PM quartiles)")
    results['mac_vs_aeth_pm'] = plot_mac_vs_aethalometer_pm_quartiles(comprehensive_df)
    
    print(f"\n" + "="*60)
    print("ANALYSIS 7: Correlation Matrix")
    results['correlation_matrix'] = create_correlation_matrix(comprehensive_df)
    
    print(f"\n" + "="*60)
    print("ADDITIONAL ANALYSIS 2: Time-series Ribbon Plot")
    results['timeseries_ribbons'] = create_timeseries_ribbon_plot(comprehensive_df)
    
    print(f"\n" + "="*60)
    print("ADDITIONAL ANALYSIS 7: Comprehensive Correlation Heat-map")
    results['comprehensive_heatmap'] = create_comprehensive_correlation_heatmap(comprehensive_df)
    
    print(f"\n✅ ALL ADDITIONAL ANALYSES COMPLETE!")
    
    return results, comprehensive_df

# %% [markdown]
# ## Example Usage
# 
# Run the analyses using your existing data:

# %%
# Example usage (uncomment and modify as needed):

# # Method 1: If you have the data already loaded
# # additional_results, comprehensive_dataset = run_additional_analyses(
# #     aethalometer_df=your_aethalometer_df,
# #     overlap_df_all=your_overlap_df_all,
# #     merged_data=your_merged_data
# # )

# # Method 2: Load fresh data (modify paths)
# if False:  # Set to True to run data loading
     # Load aethalometer data
     from your_main_script import load_aethalometer_data
     aethalometer_df = load_aethalometer_data(AETHALOMETER_PATH, convert_to_ug=True)
    
     # Load and process overlap data
     from your_main_script import load_filter_sample_data, find_overlapping_excellent_data
     etad_data, ftir_data = load_filter_sample_data(DB_PATH)
     # ... additional processing steps
    
     # Run analyses
     additional_results, comprehensive_dataset = run_additional_analyses(
         aethalometer_df, overlap_df_all, merged_data
     )

print("📝 TO RUN THE ANALYSES:")
print("1. Make sure your data is loaded (aethalometer_df, overlap_df_all, merged_data)")
print("2. Run: additional_results, comprehensive_dataset = run_additional_analyses(aethalometer_df, overlap_df_all, merged_data)")
print("\nThis will create all 6 requested analyses plus the time-series ribbon plot and comprehensive correlation heatmap!")

# %% [markdown]
# ## Important Considerations for BC Method Comparisons
# 
# Based on recent research findings correlations between aethalometer, FTIR, and HIPS measurements can vary significantly depending on several factors:
# 
# **Instrument-Specific Factors:**
# - Dual-spot aethalometers (AE33) provide real-time loading compensation, improving accuracy compared to older models
# - Aethalometer values can be only 23% of integrating sphere values for pure BC aerosols, indicating calibration dependencies
# - Different aethalometer models can show ~9% differences in BC concentrations
# 
# **Environmental Dependencies:**
# - Correlation slopes between thermal EC and optical BC varied from 3.3 to 2.7 in different cities, and dropped to 0.4 during forest fire events
# - Absorption Ångström exponents significantly affect source apportionment results and correlations
# - Seasonal variations in aerosol composition affect spectral dependence of absorption
# 
# **Interpretation Guidelines:**
# 1. **Expected correlation ranges**: r = 0.6-0.9 for well-calibrated instruments under stable conditions
# 2. **Seasonal effects**: Expect lower correlations during biomass burning episodes or dust events
# 3. **Loading effects**: Monitor for systematic biases, especially at high concentrations
# 4. **Wavelength dependence**: Red channel (635/660nm) typically shows best correlation with thermal methods
# 
# # %% [markdown]
# ## Summary
# 
# This notebook provides:
# 
# **Core Analyses (4-7):**
# - **Analysis 4**: HIPS Fabs vs Aethalometer (both attenuation and BCc)
# - **Analysis 5**: FTIR-BC vs Aethalometer (both attenuation and BCc)  
# - **Analysis 6**: HIPS-derived MAC vs Aethalometer BCc (colored by PM quartiles)
# - **Analysis 7**: Key variable correlation matrix
# 
# **Additional Analyses:**
# - **Time-series ribbon plot**: Daily BC estimates with ±1σ bands
# - **Comprehensive correlation heatmap**: All elements + ions + BC estimates
# 
# **Features:**
# - Seasonal color coding where available
# - Statistical reporting (correlations, p-values)
# - Robust data handling and error checking
# - Publication-quality visualizations
# - Comprehensive logging and status updates
# - Literature-based interpretation guidelines
# 
# **Key Questions Addressed:**
# - How do different BC measurement methods correlate?
# - What role does PM mass loading play in MAC variability?
# - Are there seasonal patterns in method agreement?
# - Which chemical species are most strongly correlated?
# - When do methods agree/disagree over time?
# - What are reasonable correlation expectations based on current literature?


IndentationError: unexpected indent (1497778441.py, line 78)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from scipy.stats import pearsonr
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.titlesize': 16
})

# %%
# NOTE: Update these paths to match your data locations
AETHALOMETER_PATH = "/path/to/your/aethalometer/data.csv"
DB_PATH = "/path/to/your/spartan_ftir_hips.db"
SPECIATION_CSV_PATH = "/path/to/your/FilterBased_ChemSpecPM25_ETAD.csv"

print("📋 ADDITIONAL ANALYSIS NOTEBOOK")
print("="*60)
print("This notebook requires your existing data to be loaded first.")
print("Make sure to run your main analysis scripts to populate:")
print("- aethalometer_df")
print("- overlap_df_all") 
print("- merged_data")
print("- results dictionary")

📋 ADDITIONAL ANALYSIS NOTEBOOK
This notebook requires your existing data to be loaded first.
Make sure to run your main analysis scripts to populate:
- aethalometer_df
- overlap_df_all
- merged_data
- results dictionary


In [4]:
# ## Data Loading and Preparation
# 
# Load data using existing functions from your main analysis

# %%
def prepare_comprehensive_dataset(aethalometer_df, overlap_df_all, merged_data):
    """
    Prepare a comprehensive dataset combining all three measurement methods
    """
    print("🔗 Preparing comprehensive dataset...")
    
    # Start with the overlap data (has excellent periods)
    comprehensive_df = overlap_df_all.copy()
    
    # Add aethalometer attenuation data if available
    aethalometer_columns = ['Red ATN1', 'Red ATN2', 'Red BCc', 'Blue BCc', 'Green BCc', 'IR BCc', 'UV BCc']
    available_aeth_cols = [col for col in aethalometer_columns if col in aethalometer_df.columns]
    
    print(f"   Available aethalometer columns: {available_aeth_cols}")
    
    # Map overlap periods to aethalometer data
    for i, row in comprehensive_df.iterrows():
        period_start = row['start_time']
        period_end = row['end_time']
        
        # Extract aethalometer data for this period
        period_aeth = aethalometer_df.loc[period_start:period_end]
        
        if len(period_aeth) > 0:
            # Calculate statistics for each available column
            for col in available_aeth_cols:
                if col in period_aeth.columns:
                    col_data = period_aeth[col].dropna()
                    if len(col_data) > 0:
                        comprehensive_df.loc[i, f'aeth_{col.replace(" ", "_").lower()}_mean'] = col_data.mean()
                        comprehensive_df.loc[i, f'aeth_{col.replace(" ", "_").lower()}_std'] = col_data.std()
                        comprehensive_df.loc[i, f'aeth_{col.replace(" ", "_").lower()}_median'] = col_data.median()
    
    # Add chemical speciation data from merged_data if available
    speciation_columns = ['PM25_mass', 'Iron', 'Potassium_Ion', 'Sulfate_Ion', 'Nitrate_Ion', 
                         'Ammonium_Ion', 'Aluminum', 'Manganese', 'Zinc', 'Copper']
    
    # Try to match dates between comprehensive_df and merged_data
    if merged_data is not None and len(merged_data) > 0:
        print("   Adding chemical speciation data...")
        
        for spec_col in speciation_columns:
            if spec_col in merged_data.columns:
                comprehensive_df[spec_col] = np.nan
                
                # Match by filter_id if available
                if 'filter_id' in comprehensive_df.columns and 'filter_id' in merged_data.columns:
                    for i, row in comprehensive_df.iterrows():
                        if pd.notna(row['filter_id']):
                            matching_spec = merged_data[merged_data['filter_id'] == row['filter_id']]
                            if len(matching_spec) > 0:
                                comprehensive_df.loc[i, spec_col] = matching_spec[spec_col].iloc[0]
                
                # If no filter_id match, try date matching
                else:
                    for i, row in comprehensive_df.iterrows():
                        sample_date = row['start_time'].date()
                        
                        # Try different date columns in merged_data
                        date_cols = ['sample_date', 'Start_Date', 'date']
                        for date_col in date_cols:
                            if date_col in merged_data.columns:
                                matching_spec = merged_data[merged_data[date_col].dt.date == sample_date]
                                if len(matching_spec) > 0:
                                    comprehensive_df.loc[i, spec_col] = matching_spec[spec_col].iloc[0]
                                    break
    
    print(f"✅ Comprehensive dataset prepared: {len(comprehensive_df)} samples")
    print(f"   Columns: {len(comprehensive_df.columns)}")
    
    return comprehensive_df

In [5]:
# Example of how to load data (adjust based on your actual data loading)
# This assumes you've already run your main analysis and have the data loaded

# If you need to load fresh data, uncomment and modify these lines:
# from your_main_analysis_script import load_aethalometer_data, load_filter_sample_data, merge_etad_datasets
# aethalometer_df = load_aethalometer_data(AETHALOMETER_PATH, convert_to_ug=True)
# etad_data, ftir_data = load_filter_sample_data(DB_PATH)
# merged_data = merge_etad_datasets(ftir_data, speciation_data)

# For now, we'll assume data is already loaded
print("📊 Assuming data is already loaded from main analysis...")
print("   Variables expected: aethalometer_df, overlap_df_all, merged_data")

# %% [markdown]
# ## Analysis 4: HIPS Fabs vs Aethalometer Attenuation/BCc

# %%
def plot_hips_vs_aethalometer(comprehensive_df, use_attenuation=True, wavelength='red'):
    """
    Plot HIPS Fabs vs Aethalometer attenuation or BCc
    
    Parameters:
    -----------
    comprehensive_df : pandas.DataFrame
        Combined dataset
    use_attenuation : bool
        If True, use ATN1; if False, use BCc
    wavelength : str
        Wavelength to use ('red', 'blue', 'green', 'ir', 'uv')
    """
    
    print(f"📈 Analysis 4: HIPS Fabs vs Aethalometer {'Attenuation' if use_attenuation else 'BCc'}")
    
    # Determine which aethalometer column to use
    if use_attenuation:
        aeth_col = f'aeth_{wavelength}_atn1_mean'
        aeth_label = f'{wavelength.title()} ATN1'
        aeth_units = 'ATN units'
    else:
        aeth_col = f'aeth_{wavelength}_bcc_mean'
        aeth_label = f'{wavelength.title()} BCc'
        aeth_units = 'μg/m³'
    
    # Check if columns exist
    if aeth_col not in comprehensive_df.columns:
        print(f"❌ Column {aeth_col} not found. Available columns:")
        available = [col for col in comprehensive_df.columns if 'aeth_' in col]
        for col in available[:10]:  # Show first 10
            print(f"   - {col}")
        return None
    
    if 'Fabs' not in comprehensive_df.columns:
        print(f"❌ Fabs column not found")
        return None
    
    # Prepare data
    df_clean = comprehensive_df[[aeth_col, 'Fabs']].dropna()
    
    if len(df_clean) < 5:
        print(f"❌ Insufficient data: only {len(df_clean)} samples")
        return None
    
    x_data = df_clean[aeth_col]
    y_data = df_clean['Fabs']
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Check for seasonal data
    if 'season' in comprehensive_df.columns:
        seasons_data = comprehensive_df.loc[df_clean.index, 'season'].dropna()
        
        if len(seasons_data) > 0:
            season_colors = {
                'Dry Season (Bega)': '#d35400', 
                'Belg Rainy Season': '#27ae60', 
                'Kiremt Rainy Season': '#2980b9'
            }
            
            print(f"   🌍 Adding seasonal color coding")
            
            for season in seasons_data.unique():
                if pd.isna(season):
                    continue
                    
                season_mask = comprehensive_df.loc[df_clean.index, 'season'] == season
                if season_mask.sum() < 3:
                    continue
                
                season_x = x_data[season_mask]
                season_y = y_data[season_mask]
                
                if season in season_colors and len(season_x) >= 3:
                    ax.scatter(season_x, season_y, 
                              color=season_colors[season], alpha=0.7, s=60, 
                              label=season.split()[0], edgecolors='black', linewidth=1)
                    
                    # Individual regression line
                    if len(season_x) >= 5:
                        z_season = np.polyfit(season_x, season_y, 1)
                        p_season = np.poly1d(z_season)
                        x_line_season = np.linspace(season_x.min(), season_x.max(), 50)
                        ax.plot(x_line_season, p_season(x_line_season), 
                               color=season_colors[season], linewidth=2, 
                               linestyle='--', alpha=0.9)
        else:
            # No seasonal data
            ax.scatter(x_data, y_data, alpha=0.7, s=60, color='blue', 
                      edgecolors='black', linewidth=1)
    else:
        # No seasonal column
        ax.scatter(x_data, y_data, alpha=0.7, s=60, color='blue', 
                  edgecolors='black', linewidth=1)
    
    # Overall regression line
    if len(x_data) > 5:
        z_overall = np.polyfit(x_data, y_data, 1)
        p_overall = np.poly1d(z_overall)
        x_line_overall = np.linspace(x_data.min(), x_data.max(), 100)
        ax.plot(x_line_overall, p_overall(x_line_overall), 'black', linewidth=3, 
               linestyle='-', alpha=0.8, label='Overall Trend')
        
        # Statistics
        r_overall, p_overall = pearsonr(x_data, y_data)
        ax.text(0.05, 0.95, f'r = {r_overall:.3f}\np = {p_overall:.2e}\nn = {len(x_data)}', 
               transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               fontweight='bold', va='top')
    
    ax.set_xlabel(f'Aethalometer {aeth_label} ({aeth_units})', fontweight='bold')
    ax.set_ylabel('HIPS Fabs (Mm⁻¹)', fontweight='bold')
    ax.set_title(f'HIPS Fabs vs Aethalometer {aeth_label}\n(n={len(x_data)} samples)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if 'season' in comprehensive_df.columns:
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    return r_overall if 'r_overall' in locals() else None

📊 Assuming data is already loaded from main analysis...
   Variables expected: aethalometer_df, overlap_df_all, merged_data


In [6]:
# ## Analysis 5: FTIR-BC vs Aethalometer Attenuation/BCc

# %%
def plot_ftir_vs_aethalometer(comprehensive_df, use_attenuation=True, wavelength='red'):
    """
    Plot FTIR-BC vs Aethalometer attenuation or BCc
    """
    
    print(f"📈 Analysis 5: FTIR-BC vs Aethalometer {'Attenuation' if use_attenuation else 'BCc'}")
    
    # Determine which aethalometer column to use
    if use_attenuation:
        aeth_col = f'aeth_{wavelength}_atn1_mean'
        aeth_label = f'{wavelength.title()} ATN1'
        aeth_units = 'ATN units'
    else:
        aeth_col = f'aeth_{wavelength}_bcc_mean'
        aeth_label = f'{wavelength.title()} BCc'
        aeth_units = 'μg/m³'
    
    # Check for FTIR-BC column
    ftir_col = 'EC_FTIR'
    if ftir_col not in comprehensive_df.columns:
        print(f"❌ {ftir_col} column not found")
        return None
    
    if aeth_col not in comprehensive_df.columns:
        print(f"❌ {aeth_col} column not found")
        return None
    
    # Prepare data
    df_clean = comprehensive_df[[aeth_col, ftir_col]].dropna()
    
    if len(df_clean) < 5:
        print(f"❌ Insufficient data: only {len(df_clean)} samples")
        return None
    
    x_data = df_clean[aeth_col]
    y_data = df_clean[ftir_col]
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Check for seasonal data
    if 'season' in comprehensive_df.columns:
        seasons_data = comprehensive_df.loc[df_clean.index, 'season'].dropna()
        
        if len(seasons_data) > 0:
            season_colors = {
                'Dry Season (Bega)': '#d35400', 
                'Belg Rainy Season': '#27ae60', 
                'Kiremt Rainy Season': '#2980b9'
            }
            
            print(f"   🌍 Adding seasonal color coding")
            
            for season in seasons_data.unique():
                if pd.isna(season):
                    continue
                    
                season_mask = comprehensive_df.loc[df_clean.index, 'season'] == season
                if season_mask.sum() < 3:
                    continue
                
                season_x = x_data[season_mask]
                season_y = y_data[season_mask]
                
                if season in season_colors and len(season_x) >= 3:
                    ax.scatter(season_x, season_y, 
                              color=season_colors[season], alpha=0.7, s=60, 
                              label=season.split()[0], edgecolors='black', linewidth=1)
                    
                    # Individual regression line
                    if len(season_x) >= 5:
                        z_season = np.polyfit(season_x, season_y, 1)
                        p_season = np.poly1d(z_season)
                        x_line_season = np.linspace(season_x.min(), season_x.max(), 50)
                        ax.plot(x_line_season, p_season(x_line_season), 
                               color=season_colors[season], linewidth=2, 
                               linestyle='--', alpha=0.9)
        else:
            ax.scatter(x_data, y_data, alpha=0.7, s=60, color='red', 
                      edgecolors='black', linewidth=1)
    else:
        ax.scatter(x_data, y_data, alpha=0.7, s=60, color='red', 
                  edgecolors='black', linewidth=1)
    
    # Overall regression line
    if len(x_data) > 5:
        z_overall = np.polyfit(x_data, y_data, 1)
        p_overall = np.poly1d(z_overall)
        x_line_overall = np.linspace(x_data.min(), x_data.max(), 100)
        ax.plot(x_line_overall, p_overall(x_line_overall), 'black', linewidth=3, 
               linestyle='-', alpha=0.8, label='Overall Trend')
        
        # Statistics
        r_overall, p_overall = pearsonr(x_data, y_data)
        ax.text(0.05, 0.95, f'r = {r_overall:.3f}\np = {p_overall:.2e}\nn = {len(x_data)}', 
               transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               fontweight='bold', va='top')
    
    ax.set_xlabel(f'Aethalometer {aeth_label} ({aeth_units})', fontweight='bold')
    ax.set_ylabel('FTIR-BC (μg/m³)', fontweight='bold')
    ax.set_title(f'FTIR-BC vs Aethalometer {aeth_label}\n(n={len(x_data)} samples)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if 'season' in comprehensive_df.columns:
        ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    return r_overall if 'r_overall' in locals() else None

In [7]:
# %% [markdown]
# ## Analysis 6: HIPS-derived MAC vs Aethalometer BCc (colored by PM quartiles)

# %%
def plot_mac_vs_aethalometer_pm_quartiles(comprehensive_df, wavelength='red'):
    """
    Plot HIPS-derived MAC vs Aethalometer BCc, colored by PM loading quartiles
    """
    
    print(f"📈 Analysis 6: HIPS-derived MAC vs Aethalometer BCc (colored by PM quartiles)")
    
    aeth_col = f'aeth_{wavelength}_bcc_mean'
    
    # Check required columns
    required_cols = [aeth_col, 'Fabs', 'EC_FTIR']
    missing_cols = [col for col in required_cols if col not in comprehensive_df.columns]
    
    if missing_cols:
        print(f"❌ Missing required columns: {missing_cols}")
        return None
    
    # Calculate HIPS-derived MAC
    comprehensive_df = comprehensive_df.copy()
    comprehensive_df['HIPS_MAC'] = comprehensive_df['Fabs'] / comprehensive_df['EC_FTIR']
    
    # Prepare data (filter reasonable MAC values)
    df_clean = comprehensive_df[[aeth_col, 'HIPS_MAC', 'PM25_mass']].dropna()
    reasonable_mask = (df_clean['HIPS_MAC'] > 5) & (df_clean['HIPS_MAC'] < 25)
    df_clean = df_clean[reasonable_mask]
    
    if len(df_clean) < 10:
        print(f"❌ Insufficient data: only {len(df_clean)} samples")
        return None
    
    # Create PM2.5 quartiles
    pm_quartiles = pd.qcut(df_clean['PM25_mass'], q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
    
    x_data = df_clean[aeth_col]
    y_data = df_clean['HIPS_MAC']
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Color by PM quartiles
    quartile_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
    
    print(f"   📊 PM2.5 quartiles:")
    for i, quartile in enumerate(pm_quartiles.cat.categories):
        quartile_mask = pm_quartiles == quartile
        quartile_data = df_clean[quartile_mask]
        
        if len(quartile_data) >= 3:
            quartile_x = quartile_data[aeth_col]
            quartile_y = quartile_data['HIPS_MAC']
            pm_range = f"{quartile_data['PM25_mass'].min():.1f}-{quartile_data['PM25_mass'].max():.1f}"
            
            ax.scatter(quartile_x, quartile_y, 
                      color=quartile_colors[i], alpha=0.7, s=60, 
                      label=f'{quartile}\n({pm_range} μg/m³)', 
                      edgecolors='black', linewidth=1)
            
            print(f"     {quartile}: {pm_range} μg/m³, n={len(quartile_data)}")
            
            # Individual regression line for each quartile
            if len(quartile_x) >= 5:
                z_quartile = np.polyfit(quartile_x, quartile_y, 1)
                p_quartile = np.poly1d(z_quartile)
                x_line_quartile = np.linspace(quartile_x.min(), quartile_x.max(), 50)
                ax.plot(x_line_quartile, p_quartile(x_line_quartile), 
                       color=quartile_colors[i], linewidth=2, 
                       linestyle='--', alpha=0.8)
    
    # Overall regression line
    if len(x_data) > 5:
        z_overall = np.polyfit(x_data, y_data, 1)
        p_overall = np.poly1d(z_overall)
        x_line_overall = np.linspace(x_data.min(), x_data.max(), 100)
        ax.plot(x_line_overall, p_overall(x_line_overall), 'black', linewidth=3, 
               linestyle='-', alpha=0.8, label='Overall Trend')
        
        # Statistics
        r_overall, p_overall = pearsonr(x_data, y_data)
        ax.text(0.05, 0.95, f'Overall: r = {r_overall:.3f}\np = {p_overall:.2e}\nn = {len(x_data)}', 
               transform=ax.transAxes,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
               fontweight='bold', va='top')
    
    ax.set_xlabel(f'Aethalometer {wavelength.title()} BCc (μg/m³)', fontweight='bold')
    ax.set_ylabel('HIPS-derived MAC (m²/g)', fontweight='bold')
    ax.set_title(f'HIPS MAC vs Aethalometer BCc\n(Colored by PM₂.₅ Loading Quartiles)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
    return r_overall if 'r_overall' in locals() else None


In [8]:
# ## Analysis 7: Correlation Matrix

# %%
def create_correlation_matrix(comprehensive_df):
    """
    Create correlation matrix of key variables
    """
    
    print(f"📊 Analysis 7: Correlation Matrix")
    
    # Define key variables for correlation matrix
    key_variables = []
    variable_labels = {}
    
    # Aethalometer variables
    aeth_vars = ['aeth_red_atn1_mean', 'aeth_red_bcc_mean', 'aeth_blue_bcc_mean', 'aeth_green_bcc_mean']
    for var in aeth_vars:
        if var in comprehensive_df.columns:
            key_variables.append(var)
            if 'atn1' in var:
                variable_labels[var] = 'Aeth ATN1'
            elif 'red_bcc' in var:
                variable_labels[var] = 'Aeth Red BCc'
            elif 'blue_bcc' in var:
                variable_labels[var] = 'Aeth Blue BCc'
            elif 'green_bcc' in var:
                variable_labels[var] = 'Aeth Green BCc'
    
    # HIPS and FTIR variables
    if 'Fabs' in comprehensive_df.columns:
        key_variables.append('Fabs')
        variable_labels['Fabs'] = 'HIPS Fabs'
    
    if 'EC_FTIR' in comprehensive_df.columns:
        key_variables.append('EC_FTIR')
        variable_labels['EC_FTIR'] = 'FTIR-BC'
    
    # Chemical species
    chem_vars = ['Iron', 'Potassium_Ion', 'PM25_mass', 'Sulfate_Ion', 'Aluminum']
    for var in chem_vars:
        if var in comprehensive_df.columns:
            key_variables.append(var)
            if var == 'Potassium_Ion':
                variable_labels[var] = 'K⁺'
            elif var == 'PM25_mass':
                variable_labels[var] = 'PM₂.₅ Mass'
            elif var == 'Sulfate_Ion':
                variable_labels[var] = 'SO₄²⁻'
            else:
                variable_labels[var] = var
    
    if len(key_variables) < 3:
        print(f"❌ Insufficient variables for correlation matrix: {key_variables}")
        return None
    
    print(f"   Variables for correlation matrix: {len(key_variables)}")
    for var in key_variables:
        print(f"     - {variable_labels.get(var, var)}")
    
    # Calculate correlation matrix
    corr_data = comprehensive_df[key_variables].dropna()
    
    if len(corr_data) < 5:
        print(f"❌ Insufficient complete cases: {len(corr_data)}")
        return None
    
    corr_matrix = corr_data.corr()
    
    # Create heatmap
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Use variable labels for display
    display_labels = [variable_labels.get(var, var) for var in key_variables]
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Hide upper triangle
    
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                square=True, fmt='.3f', cbar_kws={"shrink": .8},
                xticklabels=display_labels, yticklabels=display_labels, ax=ax)
    
    ax.set_title(f'Correlation Matrix (R values)\n(n={len(corr_data)} complete cases)', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return corr_matrix